In [1]:
import json
from langchain_core.documents import Document

## 1. Creating LangChain Documents

In [2]:
def load_rules(path):
    """Load the rule cards created in notebook 01."""
    with open(path) as f:
        return json.load(f)

def rules_to_documents(rules):
    """Turn each rule card into a LangChain Document."""
    output = []
    for rule in rules:
        output.append(
            
            Document(
                page_content=rule["text"],
                metadata={
                    "rule_id": rule["rule_id"],
                    "title": rule["title"],
                    "updated": rule["updated"],
                    "chapter": rule["chapter"],
                    "start_page": rule["start_page"],
                    "end_page": rule["end_page"],
                    "printed_start": rule["printed_start"],
                    "printed_end": rule["printed_end"],
                }
            )

        )

    return output

rules = load_rules("../data/processed/b3_rules.json")
documents = rules_to_documents(rules)

print(len(documents))
print(documents[0].metadata)
print(documents[0].page_content[:200])

123
{'rule_id': 'B3-1-01', 'title': 'Comprehensive Risk Assessment', 'updated': '11/05/2025', 'chapter': 'B3-1', 'start_page': 286, 'end_page': 288, 'printed_start': 268, 'printed_end': 269}
B3-1-01, Comprehensive Risk Assessment (11/05/2025)
Introduction
This topic contains information on the comprehensive risk assessment approach to underwriting, including:
Overview
Comprehensive Risk A


## 2. Chunking

In [3]:
from collections import Counter
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1500, chunk_overlap=200):
    """Split long documents into chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(documents)

chunks = split_documents(documents)

print("Total chunks:", len(chunks))

per_rule = Counter(c.metadata["rule_id"] for c in chunks)
print("Rules kept whole:", sum(1 for n in per_rule.values() if n == 1))
print("Most-split rules:", per_rule.most_common(3))

Total chunks: 401
Rules kept whole: 34
Most-split rules: [('B3-5.3-09', 18), ('B3-2-02', 15), ('B3-6-05', 13)]


In [4]:
dti_chunks = [c for c in chunks if c.metadata["rule_id"] == "B3-6-02"]
for i, c in enumerate(dti_chunks):
    print(f"\n--- Chunk {i} ({len(c.page_content)} chars) ---")
    print(c.page_content[:150])


--- Chunk 0 (1478 chars) ---
B3-6-02, Debt-to-Income Ratios (04/02/2025)
Introduction
This topic contains information on the use of the debt-to-income (DTI) ratio, including:
DTI 

--- Chunk 1 (1482 chars) ---
including:
cash-out refinance transactions — the maximum ratio may be lower for loan casefiles underwritten
through DU (see B2-1.3-03, Cash-Out Refina

--- Chunk 2 (1488 chars) ---
subordinate lien payments) or rental payments (see B3-6-05, Monthly Debt Obligations);
if the subject loan is a second home or investment property, us

--- Chunk 3 (1416 chars) ---
when qualifying borrowers. This is acceptable as long as Fannie Mae’s minimum requirements are met,
and lenders consistently apply the same approach t

--- Chunk 4 (1451 chars) ---
In all cases, if the lender determines that there is new subordinate financing on the subject property during the
loan process, the mortgage loan must

--- Chunk 5 (1462 chars) ---
recalculated outside of DU.
4
• If the recalculated DTI ratio exc

In [8]:
def add_chunk_headers(chunks):
    """Return new chunks with a context header and a chunk_index"""
    position = Counter()
    output = []
    for chunk in chunks:
        rule_id = chunk.metadata["rule_id"]
        header = f"Fannie Mae Selling Guide, {rule_id}: {chunk.metadata["title"]}\n\n"
        output.append(
            Document(
                page_content=header + chunk.page_content,
                metadata={**chunk.metadata, "chunk_index": position[rule_id]},
            )
        )
        position[rule_id] += 1
    return output

chunks_with_headers = add_chunk_headers(chunks)

dti = [c for c in chunks_with_headers if c.metadata["rule_id"] == "B3-6-02"]
print("\nChunk 1 metadata:", dti[1].metadata)
print("\nChunk 1 text:\n" + dti[1].page_content[:250])


Chunk 1 metadata: {'rule_id': 'B3-6-02', 'title': 'Debt-to-Income Ratios', 'updated': '04/02/2025', 'chapter': 'B3-6', 'start_page': 524, 'end_page': 528, 'printed_start': 506, 'printed_end': 509, 'chunk_index': 1}

Chunk 1 text:
Fannie Mae Selling Guide, B3-6-02: Debt-to-Income Ratios

including:
cash-out refinance transactions — the maximum ratio may be lower for loan casefiles underwritten
through DU (see B2-1.3-03, Cash-Out Refinance Transactions);
high LTV refinance tran


In [14]:
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings

# Load the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)

def first_chunk(rule_id):
    """Return the first chunk of a given rule."""
    return next(
        c for c in chunks_with_headers
        if c.metadata["rule_id"] == rule_id and c.metadata["chunk_index"] == 0
    )

def compare(question, samples):
    """Print how similar a question is to each sample chunk."""
    question_vector = embedding_model.embed_query(question)
    chunk_vectors = embedding_model.embed_documents([c.page_content for c in samples])
    print(f"\nQuestion: {question}")
    for chunk, vector in zip(samples, chunk_vectors):
        similarity = np.dot(question_vector, vector)
        print(f"  {similarity:.3f}  {chunk.metadata['rule_id']}: {chunk.metadata['title']}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [15]:
samples = [
    first_chunk("B3-6-02"),     # Debt-to-Income Ratios
    first_chunk("B3-5.1-01"),   # Credit Scores
    first_chunk("B3-4.3-04"),   # Personal Gifts
]

# What does an embedding look like?
vector = embedding_model.embed_query("test")
print("Embedding length:", len(vector))
print("First 5 numbers:", np.round(vector[:5], 3))

# Compare two different questions against the same three chunks
compare("How much debt can a borrower have compared to their income?", samples)
compare("Can my parents help me with the down payment?", samples)

Embedding length: 384
First 5 numbers: [-0.031  0.024  0.031 -0.006 -0.029]

Question: How much debt can a borrower have compared to their income?
  0.693  B3-6-02: Debt-to-Income Ratios
  0.605  B3-5.1-01: General Requirements for Credit Scores
  0.621  B3-4.3-04: Personal Gifts

Question: Can my parents help me with the down payment?
  0.559  B3-6-02: Debt-to-Income Ratios
  0.551  B3-5.1-01: General Requirements for Credit Scores
  0.617  B3-4.3-04: Personal Gifts
